# Olist 电商数据分析

本 Notebook 使用 MySQL 数据和 Python 工具链，对 Olist 巴西电商数据进行清洗、分析与可视化。

分析内容覆盖订单、客户、商品、卖家、营销线索和地理位置，并通过 GMV、复购率、转化率、RFM 分群等指标，提炼可用于业务决策的结论。

## 1. 分析目标

建立一套可复用的 Olist 电商经营分析流程，输出业务指标、客户分群和可视化结论。

In [2]:
from contextlib import contextmanager
from importlib.util import find_spec
from pathlib import Path
import json
import html as html_lib
import os
import re
import uuid
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import pymysql
import yaml
from dotenv import load_dotenv
from sshtunnel import SSHTunnelForwarder
from IPython.display import HTML, Markdown, clear_output, display
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Scatter, Scatter3D
from pyecharts.globals import CurrentConfig, NotebookType

CurrentConfig.NOTEBOOK_TYPE = NotebookType.JUPYTER_LAB

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "config" / "application.yml").exists():
    raise RuntimeError("Run JupyterLab from the project root or notebooks directory.")

# Change business rules here, then rerun the analysis cells below.
PARAMS = {
    "excluded_statuses": {"canceled", "unavailable"},
    "category_top_n": 15,
    "seller_top_n": 200,
    "minimum_mql": 30,
}

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
PARAMS


{'excluded_statuses': {'canceled', 'unavailable'},
 'category_top_n': 15,
 'seller_top_n': 200,
 'minimum_mql': 30}

## 2. SSH 与 MySQL 连接

`config/application.yml` 中可以使用 `${...}` 占位符，将敏感配置从代码中分离出来，并通过 `config/.env` 注入。

首次运行前，请在 `.env` 中配置 `MYSQL_PASSWORD`；如果启用了 SSH 隧道，还需要配置 `SSH_PRIVATE_KEY_PASSWORD` 等字段。配置完成后重启 Kernel。

In [4]:
TABLE_SQL = {
    "customers": "SELECT customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state FROM customers",
    "orders": "SELECT * FROM orders",
    "order_items": "SELECT * FROM order_items",
    "order_payments": "SELECT * FROM order_payments",
    "order_reviews": "SELECT review_id, order_id, review_score, review_creation_date, review_answer_timestamp FROM order_reviews",
    "products": "SELECT * FROM products",
    "sellers": "SELECT * FROM sellers",
    "translation": "SELECT * FROM product_category_translation",
    "mql": "SELECT * FROM marketing_qualified_leads",
    "closed_deals": "SELECT * FROM closed_deals",
    "geolocation": """
        SELECT geolocation_zip_code_prefix,
               AVG(geolocation_lat) AS latitude,
               AVG(geolocation_lng) AS longitude,
               MAX(geolocation_state) AS state,
               COUNT(*) AS records
        FROM geolocation
        GROUP BY geolocation_zip_code_prefix
    """,
}

ENV_PATTERN = re.compile(r"^\$\{([A-Za-z_][A-Za-z0-9_]*)\}$")

def resolve_env(value):
    if isinstance(value, dict):
        return {key: resolve_env(item) for key, item in value.items()}
    if not isinstance(value, str):
        return value
    match = ENV_PATTERN.fullmatch(value)
    if not match:
        return value
    name = match.group(1)
    result = os.getenv(name)
    if result is None:
        raise ValueError(f"请在 config/.env 中填写 {name}。")
    return result

def as_bool(value):
    return str(value).strip().lower() in {"1", "true", "yes", "on"}

def connection_config():
    env_path = ROOT / "config" / ".env"
    yaml_path = ROOT / "config" / "application.yml"
    if not env_path.is_file():
        raise FileNotFoundError("缺少 config/.env 配置文件。")

    load_dotenv(env_path, override=True)
    with yaml_path.open(encoding="utf-8") as source:
        config = resolve_env(yaml.safe_load(source))

    mysql = config["database"]["mysql"]
    if not str(mysql.get("password", "")).strip():
        raise ValueError("请在 config/.env 的 MYSQL_PASSWORD 中填写 MySQL 密码。")

    ssh = config.get("ssh", {})
    if as_bool(ssh.get("enabled", False)):
        for field in ("host", "user", "private_key"):
            if not str(ssh.get(field, "")).strip():
                raise ValueError(f"请在 config/.env 中填写 SSH 配置：{field}。")
    return config

def mysql_config(config):
    mysql = config["database"]["mysql"]
    return {
        "host": mysql["host"],
        "port": int(mysql["port"]),
        "user": mysql["user"],
        "password": mysql["password"],
        "database": mysql["database"],
        "charset": mysql.get("charset", "utf8mb4"),
        "cursorclass": pymysql.cursors.DictCursor,
        "connect_timeout": int(mysql.get("connect_timeout", 15)),
        "read_timeout": int(mysql.get("read_timeout", 600)),
    }

@contextmanager
def mysql_endpoint():
    config = connection_config()
    mysql = mysql_config(config)
    ssh = config.get("ssh", {})

    if not as_bool(ssh.get("enabled", False)):
        yield mysql
        return

    private_key = Path(ssh["private_key"]).expanduser()
    if not private_key.is_file():
        raise FileNotFoundError(f"SSH private key not found: {private_key}")

    tunnel_options = {
        "ssh_address_or_host": (ssh["host"], int(ssh.get("port", 22))),
        "ssh_username": ssh["user"],
        "ssh_pkey": str(private_key),
        "ssh_private_key_password": ssh.get("private_key_password") or None,
        "remote_bind_address": (
            ssh.get("remote_host", "127.0.0.1"),
            int(ssh.get("remote_port", 3306)),
        ),
        "local_bind_address": ("127.0.0.1", 0),
    }
    if ssh.get("host_key"):
        tunnel_options["ssh_host_key"] = ssh["host_key"]

    tunnel = SSHTunnelForwarder(**tunnel_options)
    tunnel.start()
    try:
        mysql["host"] = "127.0.0.1"
        mysql["port"] = tunnel.local_bind_port
        yield mysql
    finally:
        tunnel.stop()

def load_mysql_data():
    frames = {}
    with mysql_endpoint() as config:
        print(f"Reading MySQL through {config['host']}:{config['port']}/{config['database']} as {config['user']}")
        with pymysql.connect(**config) as connection:
            with connection.cursor() as cursor:
                for name, sql in TABLE_SQL.items():
                    cursor.execute(sql)
                    frames[name] = pd.DataFrame(cursor.fetchall())
                    print(f"{name:16s} {len(frames[name]):>10,} rows")
    return frames

# Run once per session. Parameter changes below reuse RAW without querying MySQL again.
RAW = load_mysql_data()


Reading MySQL through 127.0.0.1:55549/olist_ecommerce as root
customers            99,440 rows
orders               99,440 rows
order_items         112,649 rows
order_payments      103,885 rows
order_reviews        99,224 rows
products             32,950 rows
sellers               3,094 rows
translation              71 rows
mql                   8,000 rows
closed_deals            842 rows
geolocation          19,015 rows


## 3. Python 数据清洗

统一处理日期、数值和邮编字段，并记录无效日期与缺失值，便于后续质量审计。

In [5]:
DATE_COLUMNS = {
    "orders": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"],
    "order_items": ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
    "mql": ["first_contact_date"],
    "closed_deals": ["won_date"],
}
NUMERIC_COLUMNS = {
    "order_items": ["order_item_id", "price", "freight_value"],
    "order_payments": ["payment_sequential", "payment_installments", "payment_value"],
    "order_reviews": ["review_score"],
    "products": ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"],
    "closed_deals": ["declared_product_catalog_size", "declared_monthly_revenue"],
    "geolocation": ["latitude", "longitude", "records"],
}
ZIP_COLUMNS = {
    "customers": "customer_zip_code_prefix",
    "sellers": "seller_zip_code_prefix",
    "geolocation": "geolocation_zip_code_prefix",
}

def clean_frame(name, frame):
    result = frame.copy()
    audit = {"table": name, "rows": len(result), "invalid_dates": 0}
    for column in result.select_dtypes(include="object"):
        result[column] = result[column].astype("string").str.strip().replace("", pd.NA)
    zip_column = ZIP_COLUMNS.get(name)
    if zip_column in result:
        result[zip_column] = result[zip_column].astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(5)
    for column in NUMERIC_COLUMNS.get(name, []):
        if column in result:
            result[column] = pd.to_numeric(result[column], errors="coerce")
    for column in DATE_COLUMNS.get(name, []):
        if column in result:
            source_not_null = result[column].notna()
            parsed = pd.to_datetime(result[column], errors="coerce")
            audit["invalid_dates"] += int((source_not_null & parsed.isna()).sum())
            result[column] = parsed
    audit["null_cells"] = int(result.isna().sum().sum())
    return result, audit

def clean_all(frames):
    cleaned, audit = {}, []
    for name, frame in frames.items():
        cleaned[name], stats = clean_frame(name, frame)
        audit.append(stats)
    return cleaned, pd.DataFrame(audit)

DATA, CLEANING_AUDIT = clean_all(RAW)
CLEANING_AUDIT


,table,rows,invalid_dates,null_cells
0,customers,99440,0,0
1,orders,99440,0,4908
2,order_items,112649,0,0
3,order_payments,103885,0,0
4,order_reviews,99224,0,0
5,products,32950,0,2448
6,sellers,3094,0,0
7,translation,71,0,0
8,mql,8000,0,60
9,closed_deals,842,0,3300


## 5. 指标构建

基于有效订单计算订单量、GMV、客单价、复购率、准时交付率和营销转化率等指标。

In [6]:
def build_analysis(data, params):
    orders = data["orders"]
    valid = orders[~orders["order_status"].isin(params["excluded_statuses"])].copy()
    valid_ids = set(valid["order_id"])
    items = data["order_items"][data["order_items"]["order_id"].isin(valid_ids)].copy()

    item_order = items.groupby("order_id", as_index=False).agg(
        item_gmv=("price", "sum"), freight_value=("freight_value", "sum"), item_count=("order_item_id", "count")
    )
    payment_order = data["order_payments"].groupby("order_id", as_index=False).agg(payment_value=("payment_value", "sum"))
    review_order = data["order_reviews"].groupby("order_id", as_index=False).agg(review_score=("review_score", "mean"))

    order_fact = (valid
        .merge(data["customers"], on="customer_id", how="left", validate="many_to_one")
        .merge(item_order, on="order_id", how="left", validate="one_to_one")
        .merge(payment_order, on="order_id", how="left", validate="one_to_one")
        .merge(review_order, on="order_id", how="left", validate="one_to_one"))
    for column in ["item_gmv", "freight_value", "item_count", "payment_value"]:
        order_fact[column] = order_fact[column].fillna(0)
    order_fact["order_month"] = order_fact["order_purchase_timestamp"].dt.to_period("M").astype("string")
    order_fact["on_time"] = order_fact["order_delivered_customer_date"] <= order_fact["order_estimated_delivery_date"]

    monthly = order_fact.groupby("order_month", as_index=False).agg(
        valid_orders=("order_id", "nunique"), item_gmv=("item_gmv", "sum"),
        payment_value=("payment_value", "sum"), unique_customers=("customer_unique_id", "nunique")
    )
    monthly["aov"] = monthly["item_gmv"] / monthly["valid_orders"]

    category = (items
        .merge(data["products"][["product_id", "product_category_name"]], on="product_id", how="left", validate="many_to_one")
        .merge(data["translation"], on="product_category_name", how="left", validate="many_to_one")
        .groupby("product_category_name_english", dropna=False, as_index=False)
        .agg(item_gmv=("price", "sum"), item_count=("order_item_id", "count"), orders=("order_id", "nunique")))
    category["category"] = category["product_category_name_english"].fillna("unknown")

    state = order_fact.groupby("customer_state", dropna=False, as_index=False).agg(
        valid_orders=("order_id", "nunique"), item_gmv=("item_gmv", "sum"),
        unique_customers=("customer_unique_id", "nunique"), payment_value=("payment_value", "sum")
    )

    seller = items.groupby("seller_id", as_index=False).agg(
        valid_orders=("order_id", "nunique"), item_count=("order_item_id", "count"),
        item_gmv=("price", "sum"), freight_value=("freight_value", "sum"))
    seller["freight_share"] = seller["freight_value"] / seller["item_gmv"].replace(0, pd.NA)

    mql = data["mql"].copy()
    closed_ids = set(data["closed_deals"]["mql_id"].dropna())
    mql["closed"] = mql["mql_id"].isin(closed_ids)
    marketing = mql.groupby("origin", dropna=False, as_index=False).agg(
        mql_count=("mql_id", "nunique"), closed_deals=("closed", "sum"))
    marketing["origin"] = marketing["origin"].fillna("unknown")
    marketing["conversion_rate"] = marketing["closed_deals"] / marketing["mql_count"]

    customer_orders = order_fact.dropna(subset=["customer_unique_id"]).copy()
    snapshot_date = customer_orders["order_purchase_timestamp"].max() + pd.Timedelta(days=1)
    rfm = customer_orders.groupby("customer_unique_id", as_index=False).agg(
        last_purchase_date=("order_purchase_timestamp", "max"),
        frequency=("order_id", "nunique"),
        monetary=("payment_value", "sum"),
    )
    rfm["recency"] = (snapshot_date - rfm["last_purchase_date"]).dt.days

    def quintile_score(values, reverse=False):
        scores = pd.qcut(values.rank(method="first"), q=5, labels=False) + 1
        return (6 - scores if reverse else scores).astype(int)

    rfm["r_score"] = quintile_score(rfm["recency"], reverse=True)
    rfm["f_score"] = quintile_score(rfm["frequency"])
    rfm["m_score"] = quintile_score(rfm["monetary"])
    rfm["rfm_score"] = rfm["r_score"].astype(str) + rfm["f_score"].astype(str) + rfm["m_score"].astype(str)
    rfm["segment"] = "Others"
    rfm.loc[(rfm["r_score"] >= 4) & (rfm["f_score"] >= 4) & (rfm["m_score"] >= 4), "segment"] = "Champions"
    rfm.loc[(rfm["r_score"] >= 3) & (rfm["f_score"] >= 4) & (rfm["segment"] == "Others"), "segment"] = "Loyal Customers"
    rfm.loc[(rfm["m_score"] >= 4) & (rfm["segment"] == "Others"), "segment"] = "Big Spenders"
    rfm.loc[(rfm["r_score"] >= 4) & (rfm["segment"] == "Others"), "segment"] = "Recent Customers"
    rfm.loc[(rfm["r_score"] <= 2) & (rfm["f_score"] >= 3) & (rfm["segment"] == "Others"), "segment"] = "At Risk"
    segment_order = ["Champions", "Loyal Customers", "Big Spenders", "Recent Customers", "At Risk", "Others"]
    rfm_summary = rfm.groupby("segment", as_index=False).agg(
        customers=("customer_unique_id", "nunique"), avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"), revenue=("monetary", "sum"),
    )
    rfm_summary["segment"] = pd.Categorical(rfm_summary["segment"], categories=segment_order, ordered=True)
    rfm_summary = rfm_summary.sort_values("segment").reset_index(drop=True)
    rfm_summary["customer_share"] = rfm_summary["customers"] / len(rfm)
    rfm_summary["revenue_share"] = rfm_summary["revenue"] / rfm["monetary"].sum()

    repeat = order_fact.groupby("customer_unique_id")["order_id"].nunique()
    comparable = order_fact.dropna(subset=["order_delivered_customer_date", "order_estimated_delivery_date"])
    kpis = pd.Series({
        "valid_orders": order_fact["order_id"].nunique(),
        "item_gmv": order_fact["item_gmv"].sum(),
        "payment_value": order_fact["payment_value"].sum(),
        "aov": order_fact["item_gmv"].sum() / order_fact["order_id"].nunique(),
        "repeat_rate": (repeat >= 2).mean(),
        "on_time_rate": comparable["on_time"].mean(),
        "mql_conversion": mql["closed"].mean(),
    })
    return {"orders": order_fact, "items": items, "monthly": monthly, "category": category,
            "state": state, "seller": seller, "marketing": marketing,
            "rfm": rfm, "rfm_summary": rfm_summary, "kpis": kpis,
            "geolocation": data["geolocation"]}

ANALYSIS = build_analysis(DATA, PARAMS)
ANALYSIS["kpis"].to_frame("value")


,value
valid_orders,"98,206.00"
item_gmv,"13,494,311.85"
payment_value,"15,738,998.97"
aov,137.41
repeat_rate,0.03
on_time_rate,0.92
mql_conversion,0.11


## 6. 可视化函数

封装月度趋势、品类、州域、营销渠道和卖家 3D 图表，支持 Pyecharts 交互式展示。

In [7]:
def _read_echarts_runtime():
    panel_spec = find_spec("panel")
    if panel_spec is None or not panel_spec.submodule_search_locations:
        raise RuntimeError("Anaconda 环境中未找到 Panel 自带的离线 ECharts 资源。")

    asset_root = Path(next(iter(panel_spec.submodule_search_locations))) / "dist" / "bundled" / "echarts"

    def read_one(pattern):
        matches = sorted(asset_root.glob(pattern))
        if not matches:
            raise FileNotFoundError(f"未找到离线图表资源：{pattern}")
        return matches[-1].read_text(encoding="utf-8")

    return read_one("echarts@*/dist/echarts.min.js"), read_one("echarts-gl@*/dist/echarts-gl.min.js")


_ECHARTS_JS, _ECHARTS_GL_JS = _read_echarts_runtime()


def display_pyechart(chart, slot="chart"):
    """将 Pyecharts 图表嵌入 Notebook 输出。"""
    html = chart.render_embed()
    external = re.compile(r'<script type="text/javascript" src="[^"]*echarts(?:-gl)?\.min\.js"></script>')
    assets = _ECHARTS_JS + (_ECHARTS_GL_JS if "echarts-gl" in chart.js_dependencies.items else "")
    html = external.sub(lambda _: f"<script>{assets}</script>", html)
    frame = html_lib.escape(html, quote=True)
    display(HTML(
        f'<iframe srcdoc="{frame}" style="width:100%;height:{chart.height};'
        'border:0;min-height:420px" loading="eager"></iframe>'
    ))

_WIDGET_OBSERVERS = globals().get("_WIDGET_OBSERVERS", {})

def replace_widget_observers(key, bindings):
    previous = _WIDGET_OBSERVERS.pop(key, [])
    for control, callback, names in previous:
        control.unobserve(callback, names=names)
    current = []
    for control, callback, names in bindings:
        control.observe(callback, names=names)
        current.append((control, callback, names))
    _WIDGET_OBSERVERS[key] = current

def zoom():
    return [opts.DataZoomOpts(type_="inside", range_start=0, range_end=100),
            opts.DataZoomOpts(type_="slider", range_start=0, range_end=100)]

def monthly_chart(frame, mode="combo"):
    x = frame["order_month"].tolist()
    orders = frame["valid_orders"].astype(int).tolist()
    gmv = frame["item_gmv"].round(2).tolist()
    if mode == "bar":
        chart = Bar().add_xaxis(x).add_yaxis("订单数", orders, label_opts=opts.LabelOpts(is_show=False))
        yaxis = opts.AxisOpts(name="订单数")
    elif mode == "line":
        chart = Line().add_xaxis(x).add_yaxis("商品 GMV", gmv, is_smooth=True, label_opts=opts.LabelOpts(is_show=False))
        yaxis = opts.AxisOpts(name="商品 GMV（BRL）")
    else:
        chart = Bar().add_xaxis(x).add_yaxis("订单数", orders, label_opts=opts.LabelOpts(is_show=False))
        chart.extend_axis(yaxis=opts.AxisOpts(name="商品 GMV（BRL）", position="right"))
        line = Line().add_xaxis(x).add_yaxis("商品 GMV", gmv, yaxis_index=1, is_smooth=True, label_opts=opts.LabelOpts(is_show=False))
        chart.overlap(line)
        yaxis = opts.AxisOpts(name="订单数")
    chart.set_global_opts(title_opts=opts.TitleOpts(title="月度订单与 GMV"), tooltip_opts=opts.TooltipOpts(trigger="axis"),
                          toolbox_opts=opts.ToolboxOpts(is_show=True), datazoom_opts=zoom(), yaxis_opts=yaxis,
                          xaxis_opts=opts.AxisOpts(name="月份", axislabel_opts=opts.LabelOpts(rotate=35)))
    return chart

def category_chart(frame, orientation="horizontal", metric="item_gmv", top_n=15):
    rows = frame.nlargest(top_n, metric)
    names, values = rows["category"].tolist(), rows[metric].round(2).tolist()
    metric_label = {"item_gmv": "商品 GMV（BRL）", "item_count": "商品件数", "orders": "订单数"}[metric]
    chart = Bar()
    if orientation == "horizontal":
        chart.add_xaxis(names[::-1]).add_yaxis(metric_label, values[::-1], label_opts=opts.LabelOpts(is_show=False)).reversal_axis()
        xaxis, yaxis = opts.AxisOpts(name=metric_label, type_="value"), opts.AxisOpts(name="商品品类")
    else:
        chart.add_xaxis(names).add_yaxis(metric_label, values, label_opts=opts.LabelOpts(is_show=False))
        xaxis, yaxis = opts.AxisOpts(name="商品品类", axislabel_opts=opts.LabelOpts(rotate=35)), opts.AxisOpts(name=metric_label)
    chart.set_global_opts(title_opts=opts.TitleOpts(title=f"商品品类 Top {top_n}：{metric_label}"),
                          tooltip_opts=opts.TooltipOpts(trigger="axis"), toolbox_opts=opts.ToolboxOpts(is_show=True),
                          datazoom_opts=zoom(), xaxis_opts=xaxis, yaxis_opts=yaxis)
    return chart

def state_chart(frame):
    points = [[int(row.valid_orders), round(row.item_gmv, 2), int(row.unique_customers), row.customer_state]
              for row in frame.itertuples()]
    chart = Scatter().add_xaxis([]).add_yaxis("州域", points, symbol_size=18, label_opts=opts.LabelOpts(is_show=False))
    chart.set_global_opts(title_opts=opts.TitleOpts(title="各州 GMV 与客户规模"),
                          tooltip_opts=opts.TooltipOpts(formatter="州域：{@[3]}<br/>订单数：{@[0]}<br/>商品 GMV：{@[1]}<br/>客户数：{@[2]}"),
                          toolbox_opts=opts.ToolboxOpts(is_show=True), datazoom_opts=zoom(),
                          xaxis_opts=opts.AxisOpts(name="订单数", type_="value"),
                          yaxis_opts=opts.AxisOpts(name="商品 GMV（BRL）", type_="value"))
    return chart

def marketing_chart(frame, mode="combo", minimum_mql=30):
    rows = frame[frame["mql_count"] >= minimum_mql].sort_values("mql_count", ascending=False)
    x = rows["origin"].tolist()
    mql = rows["mql_count"].astype(int).tolist()
    closed = rows["closed_deals"].astype(int).tolist()
    rate = (rows["conversion_rate"] * 100).round(2).tolist()
    if mode == "line":
        chart = Line().add_xaxis(x).add_yaxis("转化率（%）", rate, is_smooth=True, label_opts=opts.LabelOpts(is_show=False))
    else:
        chart = Bar().add_xaxis(x).add_yaxis("营销线索数", mql, label_opts=opts.LabelOpts(is_show=False)).add_yaxis("成交客户数", closed, label_opts=opts.LabelOpts(is_show=False))
        if mode == "combo":
            chart.extend_axis(yaxis=opts.AxisOpts(name="转化率（%）", position="right"))
            chart.overlap(Line().add_xaxis(x).add_yaxis("转化率（%）", rate, yaxis_index=1, is_smooth=True, label_opts=opts.LabelOpts(is_show=False)))
    chart.set_global_opts(title_opts=opts.TitleOpts(title="营销渠道转化率"),
                          tooltip_opts=opts.TooltipOpts(trigger="axis"), toolbox_opts=opts.ToolboxOpts(is_show=True),
                          datazoom_opts=zoom(), xaxis_opts=opts.AxisOpts(name="营销渠道", axislabel_opts=opts.LabelOpts(rotate=25)))
    return chart

def seller_3d(frame, top_n=200):
    rows = frame.nlargest(top_n, "item_gmv")
    values = [[int(row.valid_orders), round(row.item_gmv, 2), round(float(row.freight_share or 0), 4), row.seller_id]
              for row in rows.itertuples()]
    chart = Scatter3D(init_opts=opts.InitOpts(width="1100px", height="700px"))
    chart.add("卖家", values, xaxis3d_opts=opts.Axis3DOpts(name="订单数"),
              yaxis3d_opts=opts.Axis3DOpts(name="商品 GMV（BRL）"), zaxis3d_opts=opts.Axis3DOpts(name="运费占比"),
              label_opts=opts.LabelOpts(is_show=False))
    chart.set_global_opts(title_opts=opts.TitleOpts(title="卖家经营表现 3D 分析"),
                          tooltip_opts=opts.TooltipOpts(formatter="卖家：{@[3]}<br/>订单数：{@[0]}<br/>商品 GMV：{@[1]}<br/>运费占比：{@[2]}"))
    return chart

## 7. 月度 GMV / 订单 / 客户

In [ ]:
month_mode = widgets.ToggleButtons(options=[("组合图", "combo"), ("柱状图", "bar"), ("折线图", "line")], description="图表")
month_range = widgets.SelectionRangeSlider(options=ANALYSIS["monthly"]["order_month"].tolist(),
    index=(0, len(ANALYSIS["monthly"]) - 1), description="月份区间", continuous_update=False,
    layout=widgets.Layout(width="760px"))
month_output = widgets.Output()

def show_monthly(change=None):
    start, end = month_range.value
    frame = ANALYSIS["monthly"].query("@start <= order_month <= @end")
    with month_output:
        clear_output(wait=True)
        display_pyechart(monthly_chart(frame, month_mode.value), "monthly")

replace_widget_observers("monthly", [(month_mode, show_monthly, "value"), (month_range, show_monthly, "value")])
ui = widgets.VBox([month_mode, month_range])
display(ui)
display(month_output)
show_monthly()

Output()

## 8. 商品品类 / 指标 Top N

In [ ]:
category_orientation = widgets.ToggleButtons(options=[("横向", "horizontal"), ("纵向", "vertical")], description="图表")
category_metric = widgets.Dropdown(options=[("货品销售总额", "item_gmv"),("单品销售额", "item_count"),("订单数", "orders")], value="item_gmv", description="指标")
category_top = widgets.IntSlider(value=PARAMS["category_top_n"], min=5, max=50, step=5, description="Top N", continuous_update=False)
category_output = widgets.Output()

def show_category(change=None):
    with category_output:
        clear_output(wait=True)
        display_pyechart(category_chart(ANALYSIS["category"], category_orientation.value, category_metric.value, category_top.value), "category")

replace_widget_observers("category", [(control, show_category, "value") for control in [category_orientation, category_metric, category_top]])
display(widgets.VBox([category_orientation, category_metric, category_top]))
display(category_output)
show_category()

Output()

## 9. 州域与营销渠道

In [ ]:
display_pyechart(state_chart(ANALYSIS["state"]), "state")

marketing_mode = widgets.ToggleButtons(options=[("组合", "combo"), ("柱状图", "bar"), ("折线图", "line")], description="展示方式")
minimum_mql = widgets.IntSlider(value=PARAMS["minimum_mql"], min=0, max=500, step=10, description="最小线索数", continuous_update=False)
marketing_output = widgets.Output()

def show_marketing(change=None):
    with marketing_output:
        clear_output(wait=True)
        display_pyechart(marketing_chart(ANALYSIS["marketing"], marketing_mode.value, minimum_mql.value), "marketing")

replace_widget_observers("marketing", [(marketing_mode, show_marketing, "value"), (minimum_mql, show_marketing, "value")])
display(widgets.VBox([marketing_mode, minimum_mql]))
display(marketing_output)
show_marketing()

## 10. 中文字体与 Matplotlib 导出

配置可用的中文字体后，使用 Matplotlib 生成静态图，并将 PNG 文件导出到 `notebooks/exports/`。

In [ ]:
from matplotlib import font_manager
import warnings

def configure_matplotlib_chinese():
    """Select an installed CJK font so Chinese labels render correctly."""
    candidates = [
        "Microsoft YaHei",
        "Microsoft YaHei UI",
        "DengXian",
        "SimHei",
        "Noto Sans CJK SC",
        "Source Han Sans CN",
        "WenQuanYi Zen Hei",
        "Arial Unicode MS",
    ]
    available = {
        font_manager.FontProperties(fname=font_path).get_name()
        for font_path in font_manager.findSystemFonts()
    }
    selected = next((name for name in candidates if name in available), None)

    plt.rcParams["axes.unicode_minus"] = False
    if selected:
        plt.rcParams["font.family"] = "sans-serif"
        plt.rcParams["font.sans-serif"] = [selected, "DejaVu Sans"]
    else:
        warnings.warn(
            "未找到可用中文字体，中文标签可能显示为方框；"
            "请在系统中安装 Microsoft YaHei、SimHei 或 Noto Sans CJK SC。",
            RuntimeWarning,
        )
    return selected

MATPLOTLIB_FONT = configure_matplotlib_chinese()
display(Markdown("### 卖家 3D 交互图"))
display_pyechart(seller_3d(ANALYSIS["seller"], PARAMS["seller_top_n"]), "seller")

# 用独立的 Notebook 输出和显式留白，避免 3D iframe 与静态画布连在一起。
display(HTML('<div style="height:32px"></div>'))


### Matplotlib 静态图

将月度商品 GMV 导出为 PNG，便于报告和项目展示。

In [ ]:
display(Markdown("### 月度 GMV 静态图（Matplotlib）"))

monthly = ANALYSIS["monthly"]
fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
ax.plot(
    monthly["order_month"],
    monthly["item_gmv"],
    marker="o",
    linewidth=2,
    color="#2f6f9f",
)
ax.set(
    title="月度商品 GMV",
    xlabel="月份",
    ylabel="商品 GMV（BRL）",
)
ax.tick_params(axis="x", rotation=45)
ax.grid(alpha=0.25)
ax.margins(x=0.02)

EXPORT_DIR = ROOT / "notebooks" / "exports"
EXPORT_DIR.mkdir(exist_ok=True)
STATIC_EXPORT = EXPORT_DIR / "monthly_item_gmv.png"
fig.savefig(STATIC_EXPORT, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"中文字体：{MATPLOTLIB_FONT or '未找到专用中文字体'}")
print(f"静态图已导出：{STATIC_EXPORT}")


## 11. 客户 RFM 分群

以 `customer_unique_id` 为客户粒度，基于最近购买间隔（Recency）、购买频次（Frequency）和消费金额（Monetary）计算 RFM 得分，并划分客户群体。

In [ ]:
SEGMENT_LABELS = {
    "Champions": "核心客户",
    "Loyal Customers": "忠诚客户",
    "Big Spenders": "高消费客户",
    "Recent Customers": "新近客户",
    "At Risk": "流失风险客户",
    "Others": "其他客户",
}

rfm_summary = ANALYSIS["rfm_summary"].copy()
rfm_summary["segment"] = rfm_summary["segment"].astype(str)
rfm_summary_display = rfm_summary.rename(columns={
    "segment": "客户分群", "customers": "客户数", "avg_recency": "平均最近购买间隔（天）",
    "avg_frequency": "平均购买频次", "revenue": "消费金额（BRL）", "customer_share": "客户占比", "revenue_share": "消费金额占比",
}).copy()
rfm_summary_display["客户分群"] = rfm_summary_display["客户分群"].map(SEGMENT_LABELS).fillna(rfm_summary_display["客户分群"])
display(Markdown("### 客户 RFM 分群汇总"))
display(rfm_summary_display.style.format({"平均最近购买间隔（天）": "{:.1f}", "平均购买频次": "{:.2f}", "消费金额（BRL）": "{:,.2f}", "客户占比": "{:.2%}", "消费金额占比": "{:.2%}"}))
display(Markdown("### 客户 RFM 明细（前 20 名）"))
rfm_detail_display = ANALYSIS["rfm"].sort_values(["r_score", "f_score", "m_score", "monetary"], ascending=[False, False, False, False]).head(20).rename(columns={
    "customer_unique_id": "客户唯一标识", "last_purchase_date": "最近购买日期", "frequency": "购买频次", "monetary": "消费金额（BRL）",
    "recency": "最近购买间隔（天）", "r_score": "R 得分", "f_score": "F 得分", "m_score": "M 得分", "rfm_score": "RFM 得分", "segment": "客户分群",
})
rfm_detail_display["客户分群"] = rfm_detail_display["客户分群"].map(SEGMENT_LABELS).fillna(rfm_detail_display["客户分群"])
display(rfm_detail_display)
assert ANALYSIS["rfm"]["customer_unique_id"].is_unique
assert ANALYSIS["rfm"]["recency"].ge(1).all()
assert set(ANALYSIS["rfm"]["segment"]) <= set(SEGMENT_LABELS)
assert rfm_summary["customers"].sum() == len(ANALYSIS["rfm"])
assert abs(rfm_summary["revenue"].sum() - ANALYSIS["rfm"]["monetary"].sum()) < 1e-6
print(f"RFM 客户数：{len(ANALYSIS['rfm']):,}，客户分群数：{rfm_summary['segment'].nunique()}。")

## 12. 业务结论

汇总 `ANALYSIS` 中的关键指标，输出月度、品类、州域、营销渠道和客户分群的主要发现。

In [ ]:
monthly = ANALYSIS["monthly"]
category = ANALYSIS["category"]
state = ANALYSIS["state"]
marketing = ANALYSIS["marketing"]
rfm_summary = ANALYSIS["rfm_summary"]
top_month = monthly.loc[monthly["item_gmv"].idxmax()]
top_category = category.loc[category["item_gmv"].idxmax()]
top_state = state.loc[state["item_gmv"].idxmax()]
eligible = marketing[marketing["mql_count"] >= PARAMS["minimum_mql"]]
best_origin = eligible.loc[eligible["conversion_rate"].idxmax()]
largest_segment = rfm_summary.loc[rfm_summary["customers"].idxmax()]
largest_segment_name = SEGMENT_LABELS.get(largest_segment["segment"], largest_segment["segment"])
display(Markdown("\n".join([
    f"- GMV 最高月份：**{top_month['order_month']}**，GMV 为 **{top_month['item_gmv']:,.2f}**。",
    f"- GMV 最高的商品品类：**{top_category['category']}**，GMV 为 **{top_category['item_gmv']:,.2f}**。",
    f"- GMV 最高的州域：**{top_state['customer_state']}**，GMV 为 **{top_state['item_gmv']:,.2f}**。",
    f"- 客户数最多的 RFM 分群：**{largest_segment_name}**，共有 **{int(largest_segment['customers']):,}** 位客户（{largest_segment['customer_share']:.2%}）。",
    f"- 在不少于 {PARAMS['minimum_mql']} 条营销线索的渠道中，转化率最高的是 **{best_origin['origin']}**，转化率为 **{best_origin['conversion_rate']:.2%}**。",
    "- 结合订单、客户、营销和地理数据，为运营策略与 ROI 评估提供依据。",
])))